In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import sys
!{sys.executable} -m pip install context-cite

In [2]:
import accelerate
print(accelerate.__version__)

import transformers
print(transformers.__version__)


/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.10.1
4.57.1


In [3]:
import context_cite
from context_cite import ContextCiter
from transformers import AutoTokenizer

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/yuexing/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████████████████████████████| 11/11 [00:07<00:00,  1.39it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [5]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")
print("Columns in dataset:")
print(df.columns.tolist())

Columns in dataset:
['QA_ID', 'context', 'question_options', 'answer_df3', 'data_source_corr', 'Origin', '70B_sentence_ids', '70B_After_Removal', 'human_sentence_ids', 'Low_Irr_70B', 'Extra_Context_Minus_70B', 'gpt_direct_prediction', '72B_Sentence_Contents', '72B_Low_Irr']


In [8]:
import pandas as pd
import numpy as np

# ========================================
# COMPREHENSIVE DATAFRAME DIAGNOSTICS
# ========================================

print("=" * 80)
print("DATAFRAME DIAGNOSTIC REPORT")
print("=" * 80)

# Basic DataFrame Info
print("\n1. BASIC INFORMATION")
print("-" * 80)
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Column names: {list(df.columns)}")
print(f"\nDataFrame shape: {df.shape}")

# Check for required columns
print("\n2. REQUIRED COLUMNS CHECK")
print("-" * 80)
required_cols = ['context', 'question_options']
for col in required_cols:
    if col in df.columns:
        print(f"✅ Column '{col}' exists")
    else:
        print(f"❌ Column '{col}' MISSING!")

# Check for NaN/null values
print("\n3. NULL/NaN VALUES")
print("-" * 80)
print(df.isnull().sum())

# Check first 3 rows in detail
print("\n4. DETAILED CHECK OF FIRST 3 ROWS")
print("-" * 80)
for idx, row in df.head(3).iterrows():
    print(f"\n--- ROW {idx} ---")
    
    # Check context column
    if 'context' in df.columns:
        context = row['context']
        print(f"  Context:")
        print(f"    - Type: {type(context)}")
        print(f"    - Is NaN: {pd.isna(context)}")
        print(f"    - Is None: {context is None}")
        if pd.notna(context):
            print(f"    - Length: {len(str(context))}")
            print(f"    - First 100 chars: {str(context)[:100]}...")
            print(f"    - Is empty string: {str(context).strip() == ''}")
        else:
            print(f"    ⚠️ VALUE IS NaN/NULL!")
    
    # Check question_options column
    if 'question_options' in df.columns:
        question = row['question_options']
        print(f"  Question_options:")
        print(f"    - Type: {type(question)}")
        print(f"    - Is NaN: {pd.isna(question)}")
        print(f"    - Is None: {question is None}")
        if pd.notna(question):
            print(f"    - Length: {len(str(question))}")
            print(f"    - First 100 chars: {str(question)[:100]}...")
            print(f"    - Is empty string: {str(question).strip() == ''}")
        else:
            print(f"    ⚠️ VALUE IS NaN/NULL!")

# Check data types
print("\n5. DATA TYPES")
print("-" * 80)
print(df.dtypes)

# Check for empty strings
print("\n6. EMPTY STRING CHECK (first 3 rows)")
print("-" * 80)
for idx, row in df.head(3).iterrows():
    if 'context' in df.columns:
        ctx = row['context']
        if pd.notna(ctx) and str(ctx).strip() == '':
            print(f"❌ Row {idx}: 'context' is empty string")
        elif pd.isna(ctx):
            print(f"❌ Row {idx}: 'context' is NaN/NULL")
        else:
            print(f"✅ Row {idx}: 'context' looks OK")
    
    if 'question_options' in df.columns:
        q = row['question_options']
        if pd.notna(q) and str(q).strip() == '':
            print(f"❌ Row {idx}: 'question_options' is empty string")
        elif pd.isna(q):
            print(f"❌ Row {idx}: 'question_options' is NaN/NULL")
        else:
            print(f"✅ Row {idx}: 'question_options' looks OK")

# Show actual first 3 rows
print("\n7. RAW DATA PREVIEW (first 3 rows)")
print("-" * 80)
print(df.head(3))

# Check for special characters or encoding issues
print("\n8. SPECIAL CHARACTERS CHECK")
print("-" * 80)
for idx, row in df.head(3).iterrows():
    if 'context' in df.columns and pd.notna(row['context']):
        ctx_str = str(row['context'])
        non_ascii = [c for c in ctx_str if ord(c) > 127]
        if non_ascii:
            print(f"Row {idx} context has {len(non_ascii)} non-ASCII characters")
            print(f"  Examples: {non_ascii[:5]}")

# Summary
print("\n9. SUMMARY & RECOMMENDATIONS")
print("-" * 80)
issues_found = []

for idx, row in df.head(3).iterrows():
    if 'context' in df.columns:
        if pd.isna(row['context']):
            issues_found.append(f"Row {idx}: context is NaN")
        elif str(row['context']).strip() == '':
            issues_found.append(f"Row {idx}: context is empty string")
    
    if 'question_options' in df.columns:
        if pd.isna(row['question_options']):
            issues_found.append(f"Row {idx}: question_options is NaN")
        elif str(row['question_options']).strip() == '':
            issues_found.append(f"Row {idx}: question_options is empty string")

if issues_found:
    print("❌ ISSUES FOUND:")
    for issue in issues_found:
        print(f"  - {issue}")
else:
    print("✅ No obvious issues found in first 3 rows")

print("\n" + "=" * 80)
print("END OF DIAGNOSTIC REPORT")
print("=" * 80)

DATAFRAME DIAGNOSTIC REPORT

1. BASIC INFORMATION
--------------------------------------------------------------------------------
Total rows: 1297
Total columns: 14
Column names: ['QA_ID', 'context', 'question_options', 'answer_df3', 'data_source_corr', 'Origin', '70B_sentence_ids', '70B_After_Removal', 'human_sentence_ids', 'Low_Irr_70B', 'Extra_Context_Minus_70B', 'gpt_direct_prediction', '72B_Sentence_Contents', '72B_Low_Irr']

DataFrame shape: (1297, 14)

2. REQUIRED COLUMNS CHECK
--------------------------------------------------------------------------------
✅ Column 'context' exists
✅ Column 'question_options' exists

3. NULL/NaN VALUES
--------------------------------------------------------------------------------
QA_ID                        0
context                      0
question_options             0
answer_df3                   0
data_source_corr             0
Origin                       0
70B_sentence_ids             8
70B_After_Removal            8
human_sentence_ids

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# Create output directory
output_dir = "Merge Attribution Scores MedGemma27B"
os.makedirs(output_dir, exist_ok=True)

# Validation function
def validate_row(row, idx):
    """Validate that a row has valid context and question"""
    issues = []
    
    # Check if 'context' exists and is valid
    if 'context' not in row.index:
        issues.append("Missing 'context' column")
    elif pd.isna(row['context']):
        issues.append("'context' is NaN/NULL")
    elif str(row['context']).strip() == '':
        issues.append("'context' is empty string")
    
    # Check if 'question_options' exists and is valid
    if 'question_options' not in row.index:
        issues.append("Missing 'question_options' column")
    elif pd.isna(row['question_options']):
        issues.append("'question_options' is NaN/NULL")
    elif str(row['question_options']).strip() == '':
        issues.append("'question_options' is empty string")
    
    return issues

# Statistics
processed = 0
skipped = 0
errors = 0

# Loop through the first 3 rows
for idx, row in df.head(3).iterrows():
    print(f"\n{'='*60}")
    print(f"Processing row {idx}...")
    
    # Validate row data
    validation_issues = validate_row(row, idx)
    if validation_issues:
        print(f"⚠️ Skipping row {idx} due to validation issues:")
        for issue in validation_issues:
            print(f"  - {issue}")
        skipped += 1
        continue
    
    # Extract and clean data
    context_text = str(row["context"]).strip()
    question = str(row["question_options"]).strip()
    
    # Additional sanity checks
    if len(context_text) < 10:
        print(f"⚠️ Skipping row {idx}: context too short ({len(context_text)} chars)")
        skipped += 1
        continue
    
    if len(question) < 5:
        print(f"⚠️ Skipping row {idx}: question too short ({len(question)} chars)")
        skipped += 1
        continue
    
    print(f"✓ Validation passed")
    print(f"  Context length: {len(context_text)} chars")
    print(f"  Question length: {len(question)} chars")
    
    # Append instruction to query
    query_full = (
        f"{question}\n"
        "Read the question and state your answer. "
        "State your answer, starting with 'Answer:', ending with two line breaks.\n\n"
    )
    
    try:
        # Generate attribution and response
        print(f"  Generating attribution...")
        cc = ContextCiter(
            model,
            tokenizer,
            context=context_text,
            query=query_full,
            generate_kwargs={
                "max_new_tokens": 2048, 
                "do_sample": False, 
                "pad_token_id": tokenizer.eos_token_id
            }
        )
        
        # Extract model response
        raw_response = cc.response.strip()
        print(f"  Model response: {raw_response[:100]}...")
        
        match = re.search(r"Answer:\s*([A-J])", raw_response)
        extracted_answer = match.group(1).strip() if match else None
        
        if extracted_answer:
            print(f"  Extracted answer: {extracted_answer}")
        else:
            print(f"  ⚠️ Could not extract answer from response")
        
        # Get attribution scores
        result = cc.get_attributions(as_dataframe=True)
        
        if isinstance(result, pd.io.formats.style.Styler):
            result = result.data
        
        if isinstance(result, pd.DataFrame):
            # Check for NaN or infinite values
            if result.isnull().any().any():
                print(f"  ⚠️ Warning: NaN values detected in attribution, filling with 0")
                result = result.fillna(0)
            
            numeric_cols = result.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0 and np.isinf(result[numeric_cols].values).any():
                print(f"  ⚠️ Warning: Infinite values detected, replacing with 0")
                result[numeric_cols] = result[numeric_cols].replace([np.inf, -np.inf], 0)
            
            qa_id = f"Merge Q{idx + 1}"
            result["QA_ID"] = qa_id
            result["Extracted_Answer"] = extracted_answer
            
            # Save CSV
            filename = f"{qa_id}.csv".replace(" ", "_")
            filepath = os.path.join(output_dir, filename)
            result.to_csv(filepath, index=False)
            print(f"  ✅ Saved attribution to {filepath}")
            processed += 1
        else:
            print(f"  ⚠️ Unexpected result type: {type(result)}")
            errors += 1
            
    except Exception as e:
        print(f"  ❌ Error: {e}")
        errors += 1
        continue

# Final summary
print(f"\n{'='*60}")
print("PROCESSING SUMMARY")
print(f"{'='*60}")
print(f"Successfully processed: {processed}")
print(f"Skipped (validation): {skipped}")
print(f"Errors: {errors}")
print(f"Total rows attempted: {processed + skipped + errors}")
print(f"{'='*60}")


Processing row 0...
✓ Validation passed
  Context length: 578 chars
  Question length: 105 chars
  Generating attribution...
  Model response: Answer: D

<br>
<br><end_of_turn>...
  Extracted answer: D
Attributed: Answer: D

<br>
<br><end_of_turn>


  0%|                                                                             | 0/64 [00:00<?, ?it/s]/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|████████████████████████████████████████████████████████████████████| 64/64 [00:14<00:00,  4.56it/s]


  ❌ Error: Input y contains NaN.

Processing row 1...
✓ Validation passed
  Context length: 924 chars
  Question length: 236 chars
  Generating attribution...
  Model response: Answer: C

**Rationale:** The patient presents with clinical features highly suggestive of Pseudoxan...
  Extracted answer: C
Attributed: Answer: C

**Rationale:** The patient presents with clinical features highly suggestive of Pseudoxanthoma Elasticum (PXE), including thick, leathery skinfolds (especially in flexural areas like the neck, axillae, inguinal regions), yellowish papules (angioid streaks are often associated but not mentioned here), and the characteristic histopathology showing calcification of elastic fibers in the dermis (von Kossa stain). However, the patient also has a significant coagulation abnormality: low factor X activity and a prolonged prothrombin time. While PXE itself is not typically associated with coagulation defects, the combination of PXE-like skin findings and a coagulation defic

  0%|                                                                             | 0/64 [00:00<?, ?it/s]/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|████████████████████████████████████████████████████████████████████| 64/64 [00:26<00:00,  2.42it/s]


  ❌ Error: Input y contains NaN.

Processing row 2...
✓ Validation passed
  Context length: 1287 chars
  Question length: 168 chars
  Generating attribution...
